In [1]:
import os
import pandas as pd
import numpy as np
from statsforecast import StatsForecast
from statsforecast.models import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
import warnings

# Suppress annoying Nixtla/Pandas warnings to keep output clean
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# ==========================================
# 1. LOAD AND PREP DATA (WIL BE THE SAME AS XGBOOST/LSTM/PROPHET)
# ==========================================
print("Loading and processing data (same pipeline as XGBoost/LSTM)...")

#  --- 1.1Setting the paths ---
data_dir = '../preprocesing_data/processed_csv'
sales_path = os.path.join(data_dir, 'sales_data_preprocessed.csv')

# --- 1.2 Load Sales ---
sales_df = pd.read_csv(sales_path)
sales_df['Date'] = pd.to_datetime(sales_df['Date']).dt.normalize()
if 'Time' in sales_df.columns:
    sales_df = sales_df.drop(columns=['Time'])

# Aggregate to daily totals (the data is in hours, we transform it into daily version)
daily_sales = sales_df.groupby('Date').sum(numeric_only=True).reset_index()

# Identify all product columns
product_cols = [c for c in daily_sales.columns if c != 'Date']

# --- 1.2 Melt to Nixtla long format ---
df_long = pd.melt(daily_sales, id_vars=['Date'], value_vars=product_cols,
                  var_name='unique_id', value_name='y')
df_long['y'] = df_long['y'].clip(lower=0)
df_long = df_long.rename(columns={'Date': 'ds'})
df_long = df_long.sort_values(['unique_id', 'ds']).reset_index(drop=True)

# Add the 1-day lag for MASE calculation
df_long['sales_1_day_ago'] = df_long.groupby('unique_id', observed=False)['y'].shift(1)


# ==========================================
# PRODUCTS TO FORECAST
# ==========================================
# Only keep my 64 target products
# These are the 64 specific menu items I want to predict
PRODUCTS_TO_FORECAST = [
 'Avo & Hal Muffin',
 'Avo, Egg & Bacon',
 'Avo, Feta & Tom',
 'Avocado on Toast',
 'Bacon',
 'Bacon Bap',
 'Bacon Egg Brioch',
 'Bacon Waffle',
 'Baked Beans',
 'Baked Beans JP',
 'Bean Soldiers',
 'Big Breakfast',
 'Black Pudding',
 'Breakfast Hash',
 'Breakfast Muffin',
 'Breakfast Wrap',
 'Buttd Mushrooms',
 'Cheese & Bean JP',
 'Cheese JP',
 'Chick Flatbread',
 'Chicken Club',
 'Chilli Carne JP',
 'Egg Bacon Brioch',
 'Egg Bap',
 'Extra Beans',
 'F.Eggs on Toast',
 'Festive Stack',
 'Fried Egg',
 'Hash Brown',
 'Hash Brown Bites',
 'Little Avo Toast',
 'Little Bean Toas',
 'Little Egg Toast',
 'Ltle Bfast Bacon',
 'Ltle Bfast Saus',
 'Mini Hash Browns',
 'P.Eggs on Toast',
 'Poached Egg',
 'Posh Beans',
 'Roll & Butter ',
 'S.Eggs on Toast',
 'Sausage',
 'Sausage Bap',
 'Scrambled Egg',
 'Streaky Bacon',
 'Tattie Scone',
 'The Breakfast',
 'Toasted Teacake',
 'Tuna JP',
 'Tuna Mayo Mix',
 'Tuna Melt Panini',
 'Tuna Panini',
 'Tuna Toastie',
 'Veg Sausage Bap',
 'Vegan Breakfast',
 'Vegan Sausage',
 'Veggie Bap',
 'Veggie Breakfast',
 'Bakery',
 'White Toast Bread',
 'Brown Toast Bread',
 'Porridge',
 'Sourdough Toast Bread',
 'Multiseed Toast Bread',
]
df_long = df_long[df_long['unique_id'].isin(PRODUCTS_TO_FORECAST)].reset_index(drop=True)

all_products = sorted(df_long['unique_id'].unique())
print(f"Products ({len(all_products)}): {all_products}")
print(f"Date range: {df_long['ds'].min()} to {df_long['ds'].max()}")


/home/alex/miniconda3/envs/jupyterproject-project/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading and processing data (same pipeline as XGBoost/LSTM)...
Products (64): ['Avo & Hal Muffin', 'Avo, Egg & Bacon', 'Avo, Feta & Tom', 'Avocado on Toast', 'Bacon', 'Bacon Bap', 'Bacon Egg Brioch', 'Bacon Waffle', 'Baked Beans', 'Baked Beans JP', 'Bakery', 'Bean Soldiers', 'Big Breakfast', 'Black Pudding', 'Breakfast Hash', 'Breakfast Muffin', 'Breakfast Wrap', 'Brown Toast Bread', 'Buttd Mushrooms', 'Cheese & Bean JP', 'Cheese JP', 'Chick Flatbread', 'Chicken Club', 'Chilli Carne JP', 'Egg Bacon Brioch', 'Egg Bap', 'Extra Beans', 'F.Eggs on Toast', 'Festive Stack', 'Fried Egg', 'Hash Brown', 'Hash Brown Bites', 'Little Avo Toast', 'Little Bean Toas', 'Little Egg Toast', 'Ltle Bfast Bacon', 'Ltle Bfast Saus', 'Mini Hash Browns', 'Multiseed Toast Bread', 'P.Eggs on Toast', 'Poached Egg', 'Porridge', 'Posh Beans', 'Roll & Butter ', 'S.Eggs on Toast', 'Sausage', 'Sausage Bap', 'Scrambled Egg', 'Sourdough Toast Bread', 'Streaky Bacon', 'Tattie Scone', 'The Breakfast', 'Toasted Teacake', 

In [2]:
# ==========================================
# 2. SPLIT & FORECAST ALL PRODUCTS
# ==========================================
print("Training ARIMA and generating 30-day blind forecast for ALL products...")

# My train/test split - training up to Nov 1, testing Nov 2-30
train_end = pd.to_datetime('2025-11-01')
test_end = pd.to_datetime('2025-11-30')

ts_train = df_long[df_long['ds'] <= train_end].copy()
ts_test = df_long[(df_long['ds'] > train_end) & (df_long['ds'] <= test_end)].copy()

print(f"Train: {len(ts_train)} rows ({ts_train['ds'].min()} to {ts_train['ds'].max()})")
print(f"Test:  {len(ts_test)} rows ({ts_test['ds'].min()} to {ts_test['ds'].max()})")

# ARIMA(1,1,1)
model = ARIMA(order=(1,1,1), seasonal_order=(0,0,0), season_length=1, alias='ARIMA')

sf = StatsForecast(
    models=[model],
    freq='D',
    n_jobs=-1
)

# Forecast 30 days for ALL products at once
# Only pass the 3 core columns so Nixtla doesn't train on our lag
forecast_df = sf.forecast(df=ts_train[['unique_id', 'ds', 'y']], h=30)

# Merge predictions back into test set
ts_test = ts_test.merge(forecast_df, on=['unique_id', 'ds'], how='left')

# Clip negative predictions (to make sure that we don get negative predictions and shew our results)
ts_test['ARIMA'] = ts_test['ARIMA'].clip(lower=0)

print(f"Forecast generated for {ts_test['unique_id'].nunique()} products.")

Training ARIMA and generating 30-day blind forecast for ALL products...
Train: 89664 rows (2022-01-01 00:00:00 to 2025-11-01 00:00:00)
Test:  1856 rows (2025-11-02 00:00:00 to 2025-11-30 00:00:00)
Forecast generated for 64 products.


In [3]:
# ==========================================
# 3. METRIC FUNCTION - NORTH STAR METRICS
# ==========================================

# ==========================================
# Evaluation: Forecast Metrics
# ==========================================
# WAPE measures overall inventory accuracy, while MASE compares model 
# performance against a naive lag-1 baseline. MAE is used for unit-level 
# kitchen planning, and Bias identifies systematic over/under-forecasting.

def calculate_north_star_metrics(evaluation_slice):

    if len(evaluation_slice) == 0:
        return {m: 0 for m in ['WAPE', 'MASE', 'MAE', 'Bias', 'MSE', 'RMSE', 'MAPE', 'SMAPE']}

    actual_sales = evaluation_slice['y'].values.astype(float)
    predicted_sales = evaluation_slice['ARIMA'].values.astype(float)
    naive_baseline = evaluation_slice['sales_1_day_ago'].values.astype(float)

    #  NORTH STAR 1: WAPE (Weighted Absolute Percentage Error)
    # Sum of absolute errors / Sum of actual sales
    # This tells me the total % of inventory I got wrong across all products
    total_absolute_errors = np.sum(np.abs(actual_sales - predicted_sales))
    total_actual_sales = np.sum(np.abs(actual_sales))
    wape = total_absolute_errors / total_actual_sales if total_actual_sales > 0 else np.nan

    # NORTH STAR 2: MASE (Mean Absolute Scaled Error)
    # My model's MAE divided by naive baseline MAE
    # < 1.0 means my model adds value, > 1.0 means I should just use yesterday's number
    mae = mean_absolute_error(actual_sales, predicted_sales)
    naive_baseline_mae = mean_absolute_error(actual_sales, naive_baseline)
    mase = mae / naive_baseline_mae if naive_baseline_mae > 0 else np.nan

    #  NORTH STAR 3: MAE (Mean Absolute Error in units)
    # The kitchen needs to know: "how many units off will I typically be?"
    # (mae already calculated above, no need to do it again)

    # NORTH STAR 4: Bias (directional error)
    # Negative = I'm under-predicting (we'll run out of stock)
    # Positive = I'm over-predicting (we'll waste food)
    forecast_bias = np.mean(predicted_sales - actual_sales)

    #  Legacy metrics (we calculateing it , just in case we need more data for presentation)
    mse = mean_squared_error(actual_sales, predicted_sales)
    rmse = np.sqrt(mse)

    # MAPE — only on non-zero actuals to avoid division by zero
    nonzero_mask = actual_sales > 0
    mape = mean_absolute_percentage_error(actual_sales[nonzero_mask], predicted_sales[nonzero_mask]) if nonzero_mask.sum() > 0 else np.nan

    # SMAPE — symmetric version, less sensitive to zeros
    abs_errors = np.abs(predicted_sales - actual_sales)
    abs_denominator = (np.abs(actual_sales) + np.abs(predicted_sales)) / 2.0
    valid_mask = abs_denominator > 0
    smape = np.mean(abs_errors[valid_mask] / abs_denominator[valid_mask]) if valid_mask.sum() > 0 else np.nan

    return {'WAPE': wape, 'MASE': mase, 'MAE': mae, 'Bias': forecast_bias,
            'MSE': mse, 'RMSE': rmse, 'MAPE': mape, 'SMAPE': smape}

In [4]:
# ==========================================
# 4. PER-PRODUCT DETAILED SCORECARDS
# ==========================================
target_date_1 = pd.to_datetime('2025-11-02')
target_date_7 = pd.to_datetime('2025-11-08')
target_date_30 = pd.to_datetime('2025-11-30')

for product in all_products:
    p_test = ts_test[ts_test['unique_id'] == product].copy()
    p_test = p_test.sort_values('ds').reset_index(drop=True)

    df_1_day = p_test[p_test['ds'] == target_date_1]
    df_1_week = p_test[(p_test['ds'] >= target_date_1) & (p_test['ds'] <= target_date_7)]
    df_1_month = p_test[(p_test['ds'] >= target_date_1) & (p_test['ds'] <= target_date_30)]

    comparison_df = pd.DataFrame({
        'Metric': ['WAPE', 'MASE', 'MAE', 'Bias', 'MSE', 'RMSE', 'MAPE', 'SMAPE'],
        '1-Day (Nov 2)': list(calculate_north_star_metrics(df_1_day).values()),
        '1-Week (Nov 2-8)': list(calculate_north_star_metrics(df_1_week).values()),
        '1-Month (Nov 2-30)': list(calculate_north_star_metrics(df_1_month).values())
    })

    for col in comparison_df.columns[1:]:
        comparison_df[col] = comparison_df[col].apply(
            lambda x: f"{float(x):.4f}" if not np.isnan(x) else "N/A"
        )

    print(f"\n======================================================")
    print(f"  ARIMA METRICS (BLIND TEST): {product}")
    print(f"======================================================")
    print(comparison_df.to_string(index=False))

    # We round, just to be sure we dont get a 0.7 Bacon bap
    p_test['Predicted_Rounded'] = p_test['ARIMA'].round().astype(int)
    p_test['Mistake_Amount'] = (p_test['y'] - p_test['Predicted_Rounded']).abs()

    print(f"\n  THE CHECK: {product}")
    print(f"  --------------------------------------------------")
    display_df = p_test[['ds', 'y', 'Predicted_Rounded', 'Mistake_Amount']].copy()
    display_df.columns = ['Date', 'Sales', 'ARIMA_Prediction', 'Mistake_Amount']
    print(display_df.head(10).to_string(index=False))


  ARIMA METRICS (BLIND TEST): Avo & Hal Muffin
Metric 1-Day (Nov 2) 1-Week (Nov 2-8) 1-Month (Nov 2-30)
  WAPE           N/A              N/A                N/A
  MASE           N/A              N/A                N/A
   MAE        0.0000           0.0000             0.0000
  Bias        0.0000           0.0000             0.0000
   MSE        0.0000           0.0000             0.0000
  RMSE        0.0000           0.0000             0.0000
  MAPE           N/A              N/A                N/A
 SMAPE        2.0000           2.0000             2.0000

  REALITY CHECK: Avo & Hal Muffin
  --------------------------------------------------
      Date  Sales  ARIMA_Prediction  Mistake_Amount
2025-11-02      0                 0               0
2025-11-03      0                 0               0
2025-11-04      0                 0               0
2025-11-05      0                 0               0
2025-11-06      0                 0               0
2025-11-07      0                 0    

In [5]:
# ==========================================
# PER-PRODUCT NORTH STAR LEADERBOARD (sorted by Bias for kitchen staff)
# ==========================================
# I'm sorting by Bias here because that's what the kitchen actually needs to act on:
# - Products with large negative bias = we're consistently under-prepping (running out!)
# - Products with large positive bias = we're over-prepping (wasting food!)
# Then MAE tells them how many units they'll typically be off on any given day.

print("\n==================================================")
print("  PER-PRODUCT NORTH STAR METRICS")
print("  Sorted by Bias (most under-predicted first)")
print("==================================================")

target_date_1 = pd.to_datetime('2025-11-02')
target_date_30 = pd.to_datetime('2025-11-30')
per_product_metrics = []
for product_name in all_products:
    p_test = ts_test[ts_test['unique_id'] == product_name].copy()
    product_month_data = p_test[(p_test['ds'] >= target_date_1) & (p_test['ds'] <= target_date_30)]
    product_metrics = calculate_north_star_metrics(product_month_data)
    product_metrics['Product'] = product_name
    per_product_metrics.append(product_metrics)

product_leaderboard_df = pd.DataFrame(per_product_metrics)
product_leaderboard_df = product_leaderboard_df[['Product', 'WAPE', 'MASE', 'MAE', 'Bias', 'RMSE']]

# Sort by Bias so the kitchen can see which items they're under/over-prepping most
product_leaderboard_df = product_leaderboard_df.sort_values('Bias')

for col in ['WAPE', 'MASE', 'MAE', 'Bias', 'RMSE']:
    product_leaderboard_df[col] = product_leaderboard_df[col].apply(
        lambda x: f"{x:.4f}" if not np.isnan(x) else "N/A"
    )

print(product_leaderboard_df.to_string(index=False))

# Quick interpretation guide
print("\n  INTERPRETATION GUIDE:")
print("  - WAPE < 0.20 = Good (within 20% of total volume)")
print("  - MASE < 1.00 = Model beats naive baseline (yesterday\'s sales)")
print("  - Bias < 0    = Under-predicting (stock-out risk)")
print("  - Bias > 0    = Over-predicting (waste risk)")


  PER-PRODUCT NORTH STAR METRICS
  Sorted by Bias (most under-predicted first)
              Product   WAPE   MASE    MAE    Bias    RMSE
        The Breakfast 0.2917 0.7967 9.9179 -2.8430 11.8549
        Festive Stack 1.0000 1.7556 2.7241 -2.7241  3.8011
               Bakery 0.1341 0.7874 8.4445 -2.3769 11.0573
          Sausage Bap 0.3081 0.8864 3.7291 -1.6882  4.3884
            Fried Egg 0.4503 0.9345 3.8667 -1.3568  4.4585
    White Toast Bread 0.2017 0.7272 6.3438 -1.2225  9.3564
     Breakfast Muffin 0.4296 0.6709 1.6887 -0.9323  2.2423
    Brown Toast Bread 0.2311 0.7167 1.9525 -0.9035  2.5693
        Black Pudding 0.6506 0.7294 1.6601 -0.8581  2.2805
         Tattie Scone 0.7757 0.8292 1.6583 -0.7379  2.0761
              Egg Bap 0.7059 0.6177 1.1928 -0.5288  1.6483
          Baked Beans 0.4193 0.7049 2.2119 -0.4924  2.5043
        Big Breakfast 0.3942 0.8693 6.5649 -0.3693  7.8547
Multiseed Toast Bread 0.6023 0.6487 1.7446 -0.3374  2.2897
Sourdough Toast Bread 0.8854 0.7179

In [6]:
# ==========================================
# OVERALL AGGREGATE NORTH STAR METRICS
# ==========================================
# This is the "executive summary" , the monthly report, the single set of numbers I'd show to stakeholders.
# WAPE is the star here: it tells them what % of our total inventory forecast was wrong.

all_month = ts_test[(ts_test['ds'] >= target_date_1) & (ts_test['ds'] <= target_date_30)].copy()
overall_north_star = calculate_north_star_metrics(all_month)

print("\n==================================================")
print("  NORTH STAR METRICS: ARIMA (All Products)")
print("==================================================")
print(f"  WAPE:  {overall_north_star['WAPE']:.4f}  <- % of total inventory wrong")
print(f"  MASE:  {overall_north_star['MASE']:.4f}  <- {'BEATING' if overall_north_star['MASE'] < 1 else 'LOSING TO'} naive baseline")
print(f"  MAE:   {overall_north_star['MAE']:.4f}  <- avg units off per product per day")
print(f"  Bias:  {overall_north_star['Bias']:.4f}  <- {'under-predicting' if overall_north_star['Bias'] < 0 else 'over-predicting'}")
print("  ---")
for metric_name in ['RMSE', 'MAPE', 'SMAPE']:
    val = overall_north_star[metric_name]
    print(f"  {metric_name:>6s}: {val:.4f}" if not np.isnan(val) else f"  {metric_name:>6s}: N/A")


  NORTH STAR METRICS: ARIMA (All Products)
  WAPE:  0.3697  <- % of total inventory wrong
  MASE:  0.8242  <- BEATING naive baseline
  MAE:   1.6559  <- avg units off per product per day
  Bias:  -0.1378  <- under-predicting
  ---
    RMSE: 3.1945
    MAPE: 0.5409
   SMAPE: 1.0473


In [7]:
# ==========================================
# Persistence: SQLite Tracking
# ==========================================
# Use SQLite to maintain a historical record of model performance, 
# allowing for trend analysis in Tableau/PowerBI for the kitchen manager.
#
# Table 1: predictions_log — every single prediction (for actual vs predicted charts)
# Table 2: metrics_summary — aggregated scorecard (for "which model is winning?" queries)
# Also we save to CSV to make sure we got everything properly

import sqlite3
from datetime import datetime

os.makedirs('../results', exist_ok=True)

#  Generate a unique run ID so I can track this specific model run
prediction_timestamp = datetime.now()
run_id = prediction_timestamp.strftime('%Y%m%d_%H%M') + '_ARIMA'

print(f"Saving results for run: {run_id}")

#  1. Still save CSVs for quick access (backwards compatibility)
product_leaderboard_df.to_csv('../results/arima_per_product_results.csv', index=False)

print("  CSVs saved to ../results/")

#  2. SQLite Storage Framework
# Connect to (or create) my model tracking database
database_path = '../results/model_tracking.db'
db_connection = sqlite3.connect(database_path)

with db_connection:
    # Make sure i create the table if it doent exist
    db_connection.execute("""
        CREATE TABLE IF NOT EXISTS predictions_log (
            run_id TEXT,
            model_type TEXT,
            target_date TEXT,
            prediction_made_on TEXT,
            product_name TEXT,
            actual_sales REAL,
            predicted_sales REAL
        )
    """)

    db_connection.execute("""
        CREATE TABLE IF NOT EXISTS metrics_summary (
            run_id TEXT,
            model_type TEXT,
            product_name TEXT,
            evaluation_horizon TEXT,
            WAPE REAL,
            MASE REAL,
            MAE REAL,
            Bias REAL
        )
    """)

    #  Write predictions_log
    # I'm logging every single prediction so I can build actual vs predicted charts later
    predictions_for_db = ts_test[['ds', 'unique_id', 'y', 'ARIMA']].copy()
    predictions_for_db = predictions_for_db.rename(columns={
        'ds': 'target_date', 'unique_id': 'product_name',
        'y': 'actual_sales', 'ARIMA': 'predicted_sales'
    })
    predictions_for_db['run_id'] = run_id
    predictions_for_db['model_type'] = 'ARIMA'
    predictions_for_db['prediction_made_on'] = str(prediction_timestamp)
    predictions_for_db['target_date'] = predictions_for_db['target_date'].astype(str)
    predictions_for_db.to_sql('predictions_log', db_connection, if_exists='append', index=False)
    print(f"  Logged {len(predictions_for_db)} predictions to predictions_log")

    # Write metrics_summary (the scorecard)
    # This is where I can instantly query which model is "winning" across all runs

    # Define my evaluation horizons, same windows I've been using throughout
    evaluation_start = pd.to_datetime('2025-11-02')
    evaluation_week_end = pd.to_datetime('2025-11-08')
    evaluation_month_end = pd.to_datetime('2025-11-30')

    product_col_name = 'unique_id'
    horizon_products = sorted(ts_test['unique_id'].unique())

    # Build horizon slices
    month_slice = ts_test[(ts_test['ds'] >= evaluation_start) & (ts_test['ds'] <= evaluation_month_end)].copy()
    week_slice = month_slice[month_slice['ds'] <= evaluation_week_end].copy()
    day_slice = month_slice[month_slice['ds'] == evaluation_start].copy()

    horizons = {'1-Day': day_slice, '1-Week': week_slice, '1-Month': month_slice}

    metrics_rows = []

    # Per-product metrics at each horizon
    for product_name in horizon_products:
        for horizon_label, horizon_df in horizons.items():
            product_horizon_data = horizon_df[horizon_df[product_col_name] == product_name] if len(horizon_df) > 0 else horizon_df.iloc[0:0]
            if len(product_horizon_data) > 0:
                m_dict = calculate_north_star_metrics(product_horizon_data)
                metrics_rows.append({
                    'run_id': run_id,
                    'model_type': 'ARIMA',
                    'product_name': product_name,
                    'evaluation_horizon': horizon_label,
                    'WAPE': m_dict.get('WAPE', np.nan),
                    'MASE': m_dict.get('MASE', np.nan),
                    'MAE': m_dict.get('MAE', np.nan),
                    'Bias': m_dict.get('Bias', np.nan),
                })

    # ALL_PRODUCTS aggregate row for each horizon
    for horizon_label, horizon_df in horizons.items():
        if len(horizon_df) > 0:
            m_dict = calculate_north_star_metrics(horizon_df)
            metrics_rows.append({
                'run_id': run_id,
                'model_type': 'ARIMA',
                'product_name': 'ALL_PRODUCTS',
                'evaluation_horizon': horizon_label,
                'WAPE': m_dict.get('WAPE', np.nan),
                'MASE': m_dict.get('MASE', np.nan),
                'MAE': m_dict.get('MAE', np.nan),
                'Bias': m_dict.get('Bias', np.nan),
            })

    if metrics_rows:
        metrics_summary_df = pd.DataFrame(metrics_rows)
        metrics_summary_df.to_sql('metrics_summary', db_connection, if_exists='append', index=False)
        print(f"  Logged {len(metrics_summary_df)} metric rows to metrics_summary")

db_connection.close()
print(f"  SQLite database saved to {database_path}")
print(f"  Run ID: {run_id}")
print("\nDone! To query results later:")
print("  SELECT * FROM metrics_summary WHERE model_type=\'ARIMA\' ORDER BY WAPE")
print("  SELECT * FROM predictions_log WHERE run_id=\'<run_id>\' AND product_name=\'Bacon Bap\'")

Saving results for run: 20260419_1350_ARIMA
  CSVs saved to ../results/
  Logged 1856 predictions to predictions_log
  Logged 195 metric rows to metrics_summary
  SQLite database saved to ../results/model_tracking.db
  Run ID: 20260419_1350_ARIMA

Done! To query results later:
  SELECT * FROM metrics_summary WHERE model_type='ARIMA' ORDER BY WAPE
  SELECT * FROM predictions_log WHERE run_id='<run_id>' AND product_name='Bacon Bap'
